# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [1]:
!docker info

Client:
 Version:    29.8.0
 Context:    default
 Debug Mode: false
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.140.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-agent
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.31.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-ai
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.37.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-buildx
  compose: Docker Compose (Docker Inc.)
    Version:  v5.5.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-compose
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.47
    Path:     /usr/local/lib/docker/cli-plugins/docker-debug
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.4.4
    Path:     /usr/local/lib/docker/cli-plugins/docker-desktop
  dhi: CLI for managing Docker Hardened Images (Docker Inc.)
    Version:  v0.0.7
    Path:     /usr/local/lib/docker/cli-plugins/docker-dh

### What is a container?

A container is an enclosure to setup custom environments. Container can be stored as images. 

### Why do we use containers?

they isolate our process from the surrounding environment, they get build in a reproducible manner, they can be swappen in and out of a pipeline

### What is a docker image?

A docker image contains all neccessary information and data to deploy a docker container.

### Let's run our first docker image:

### Login to docker

In [2]:
# This you need to do on the command line directly

### Run your first docker container

In [3]:
!docker run hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



### Find the container ID

In [4]:
container_ids = !docker ps -a --filter ancestor=hello-world --format "{{{{.ID}}}}"
container_ids = list(container_ids)

print(container_ids)

['5601840dffd8']


### Delete the container again, give prove its deleted

In [5]:
for container_id in container_ids:
    !docker rm {container_id}

5601840dffd8


In [6]:
!docker ps -all

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

1. I would use the fastqc conda package 
2. To do it manually download the latest release from https://github.com/s-andrews/FastQC/releases/tag/v0.12.1
3. unzip it
4. use it with the gui or the command line on these files

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [7]:
# pull the container
!docker run --rm  community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

In [27]:
# run the container and save the results to a new "fastqc_results" directory
!mkdir -p fastqc_results

!docker run --rm \
    -v "$PWD/fastqc_results:/output" \
    -v "$PWD:/data" \
    community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
    fastqc \
    /data/SRX19144488_SRR23195511_1.fastq.gz \
    /data/SRX19144486_SRR23195516_1.fastq.gz \
    --outdir /output

application/gzip
application/gzip
Started analysis of SRX19144488_SRR23195511_1.fastq.gz
Approx 5% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 10% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 15% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 20% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 25% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 30% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 35% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 40% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 45% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 50% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 55% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 60% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 65% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 70% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 75% complete for SRX19144488_SRR23195511_1.fastq.gz
Approx 80% complete for SRX

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

for such a small task the gui application is easier, for streamlined pipelines the docker containers are way easier to orchestrate.

### What would you say, which approach is more reproducible?
the containers of course

### Compare the file to last weeks fastqc results, are they identical?

yes

### Is the fastqc version identical?

yes. in both cases we ran FastQC 0.12.1


## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [9]:
# open the file "my_dockerfile" in a text editor

### Explain the RUN and ENV lines you added to the file

RUN apt-get update && \
    apt-get install -y curl cowsay && \
    rm -rf /var/lib/apt/lists/*

apt-get update retrieves the current package index, apt-get install installs the apllications in the argument, -y automatically accepts installation, rm removes the index files to keep the container smaller

ENV PATH="/usr/games:${PATH}"
adds "/usr/games" to PATH -> direct access to tools inside this directory without having to navigate there

In [16]:
# build the docker image
!docker build -f my_dockerfile -t games .








[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.1s
 => => transferring dockerfile: 641B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.1s
[+] Building 0.4s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.1s
 => => transferring dockerfile: 641B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.3s
[+] Building 0.5s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.1s
 => => transferring dockerfile: 641B                                       0.0s
 => [internal] load metadata for 

In [17]:
# make sure that the image has been built
!docker images | grep games

games:latest                                                                                  a85814f58195        201MB         45.7MB        


In [22]:
# run the docker file 
!docker run --rm games cowsay hello world

 _____________
< hello world >
 -------------
        \   ^__^
         \  (oo)\_______
            (__)\       )\/\
                ||----w |
                ||     ||


## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [13]:
# use the file "salmon_docker" in this directory to build a new docker image

In [23]:
# build the image
!docker build -f salmon_docker -t salmon .








[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.1s
 => => transferring dockerfile: 417B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.1s
[+] Building 0.4s (1/2)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.1s
 => => transferring dockerfile: 417B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.3s
[+] Building 0.5s (1/2)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.1s
 => => transferring dockerfile: 417B                                       0.0s
 => [internal] load metadata for 

In [24]:
# run the docker image to give out the version of salmon
!docker run --rm salmon salmon --version

salmon 1.10.1


## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

of course not. Alot of tools come already containerized.

Find the salmon docker image online and run it on your computer.

for example this: https://hub.docker.com/r/combinelab/salmon

What is https://biocontainers.pro/ ?

In [26]:
!docker run --rm combinelab/salmon --version

salmon 2.8.0


## Are there other ways to create Docker (or Apptainer) images?

you can also use online tools for creating docker containers like the following

What is https://seqera.io/containers/ 
a tool to create docker containers of available conda and pypi packages